In [1]:
#r "C:\Users\Zver\Desktop\REPO\practice2026\practice2026\task17\bin\Debug\net9.0\task17.dll"
#r "nuget: ScottPlot, 5.0.*"
using System;
using System.Diagnostics;
using System.IO;
using System.Collections.Generic;
using ScottPlot;
using task17;
using Microsoft.DotNet.Interactive.Formatting;

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages ScottPlot, 5.0.56

Loading extensions from `C:\Users\Zver\.nuget\packages\skiasharp\2.88.9\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

In [2]:
Formatter.Register(typeof(Plot), (plotObj, writer) =>
{
    var plot = (Plot)plotObj;
    byte[] imageBytes = plot.GetImageBytes(600, 400, ImageFormat.Png);
    string base64String = Convert.ToBase64String(imageBytes);
    writer.Write($"<img src='data:image/png;base64,{base64String}' width='600' height='400' />");
}, HtmlFormatter.MimeType);

ScottPlot.Fonts.Default = "DejaVu Sans";

public class ConsoleExceptionHandler : IExceptionHandler
{
    public void Handle(Exception exception, ICommand command)
    {
        Console.WriteLine($"Ошибка при выполнении команды: {exception.Message}");
    }
}


public class RecordingTestCommand : ILongRunningCommand
{
    private readonly List<int> _log;
    private readonly int _id;
    private readonly int _maxCalls;
    private readonly CountdownEvent _completionSignal;
    private int _counter = 0;

    public RecordingTestCommand(List<int> log, int id, int maxCalls, CountdownEvent completionSignal)
    {
        _log = log;
        _id = id;
        _maxCalls = maxCalls;
        _completionSignal = completionSignal;
    }

    public bool IsCompleted => _counter >= _maxCalls;

    public void Execute()
    {
        _counter++;
        lock (_log) 
        { 
            _log.Add(_id); 
        }
        Console.WriteLine($"Поток {_id} вызов {_counter}");
        if (IsCompleted)
            _completionSignal.Signal();
    }
}

var scheduler = new RoundRobinScheduler();

var exceptionHandler = new ConsoleExceptionHandler();
var server = new ServerThread(exceptionHandler);

server.UpdateBehavior(() =>
{
    if (scheduler.HasCommand())
    {
        var cmd = scheduler.Select();
        if (cmd != null)
        {
            try
            {
                cmd.Execute();
                
                if (cmd is ILongRunningCommand longCmd && !longCmd.IsCompleted)
                {
                    scheduler.Add(cmd);
                }
            }
            catch (Exception ex)
            {
                exceptionHandler.Handle(ex, cmd);
            }
        }
    }
    else
    {
    
        Thread.Sleep(1);
    }
});


server.Start();

var executionLog = new List<int>();

var completed = new CountdownEvent(5);

for (int id = 1; id <= 5; id++)
{
    scheduler.Add(new RecordingTestCommand(executionLog, id, maxCalls: 3, completed));
}
completed.Wait();


server.QueueCommand(new HardStop(server)); 
Thread.Sleep(100);

if (!server.UnderlyingThread.Join(2000))
{
    server.UnderlyingThread.Interrupt();
    server.UnderlyingThread.Join(1000);
}


string filePath = "result.txt";
using (StreamWriter writer = new StreamWriter(filePath))
{
    writer.WriteLine("Отчет о выполнении");
    writer.WriteLine();
    writer.WriteLine("Порядок выполнения команд:");
    for (int i = 0; i < executionLog.Count; i++)
        writer.WriteLine($"{i + 1,3} -> команда {executionLog[i]}");
    writer.WriteLine();
    writer.WriteLine($"Всего вызовов: {executionLog.Count}");
    writer.WriteLine($"Команд: 5");
    writer.WriteLine($"Вызовов на команду: 3");
    writer.WriteLine($"Ожидалось: 15");
}


var plot = new Plot();
var xs = Enumerable.Range(1, executionLog.Count).Select(i => (double)i).ToArray();
var ys = executionLog.Select(id => (double)id).ToArray();

var scatter = plot.Add.Scatter(xs, ys);

scatter.Label = "Очередность вызовов";
scatter.LineStyle.Width = 2;
scatter.MarkerStyle.Size = 10;
scatter.Color = ScottPlot.Color.FromHex("#2E8B57"); 

plot.XLabel("Порядковый номер вызова Execute()");
plot.YLabel("ID команды");
plot.Title("Чередование команд в планировщике");

plot.ShowLegend(Alignment.LowerRight);

plot.SavePng("plot.png", 800, 600);

plot.Display();



(90,21): error CS0246: Не удалось найти тип или имя пространства имен "CountdownEvent" (возможно, отсутствует директива using или ссылка на сборку).

(25,22): error CS0246: Не удалось найти тип или имя пространства имен "CountdownEvent" (возможно, отсутствует директива using или ссылка на сборку).

(28,70): error CS0246: Не удалось найти тип или имя пространства имен "CountdownEvent" (возможно, отсутствует директива using или ссылка на сборку).

(81,9): error CS0103: Имя "Thread" не существует в текущем контексте.

(100,1): error CS0103: Имя "Thread" не существует в текущем контексте.



Error: compilation error